In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd

df = pd.read_csv(
    "/content/drive/MyDrive/MultilingualFakeNews/data/english_clean_v2.csv"
)

df = df[["text", "target"]]

print(df.shape)
df.head()

(588, 2)


,text,target
0,Roger Federer beats Frances Tiafoe on return t...,0
1,'It was a long time coming': Heidi Klum ends s...,1
2,Liam Payne Just Dissed Harry Styles' Solo Musi...,1
3,We all know why the right is angry at Tomi Lah...,0
4,Inside Beyoncé's baby shower Don't act like yo...,0


In [4]:
# Train / Validation split
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["target"]
)

print(train_df.shape)
print(val_df.shape)

(470, 2)
(118, 2)


In [5]:
# Loading Tokenizer
from transformers import AutoTokenizer

MODEL_NAME = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

In [6]:
# Creating dataset class
import torch
from torch.utils.data import Dataset

class FakeNewsDataset(Dataset):

    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        max_len=384
    ):

        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        text = str(self.texts[idx])

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids":
                encoding["input_ids"].flatten(),

            "attention_mask":
                encoding["attention_mask"].flatten(),

            "label":
                torch.tensor(
                    self.labels[idx],
                    dtype=torch.long
                )
        }

In [7]:
# Creating dataset objects
train_dataset = FakeNewsDataset(
    train_df["text"].tolist(),
    train_df["target"].tolist(),
    tokenizer
)

val_dataset = FakeNewsDataset(
    val_df["text"].tolist(),
    val_df["target"].tolist(),
    tokenizer
)

In [ ]:
# Creating DataLoaders
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8
)

In [9]:
# Verifying shapes

batch = next(iter(train_loader))

print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["label"].shape)

torch.Size([8, 384])
torch.Size([8, 384])
torch.Size([8])
